# Python Scripting Assignment

**Task 1:** Write Python scripts for basic file operations and data processing.
**Task 2:** Develop a simple web scraper to extract data from a website.

This notebook is fully runnable — the code cells actually execute here.

## Task 1 — File Operations & Data Processing

### 1a. Basic file operations (create, read, append, delete)

In [1]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
sample_file = data_dir / "sample.txt"

# Create / write
sample_file.write_text("line one\nline two\nline three\n")
print("Written:", sample_file.read_text())

# Append
with sample_file.open("a") as f:
    f.write("line four\n")

# Read
print("After append:\n", sample_file.read_text())

# List directory contents
print("Files in data/:", [p.name for p in data_dir.iterdir()])


Written: line one
line two
line three

After append:
 line one
line two
line three
line four

Files in data/: ['sample.txt']


### 1b. Data processing — CSV in, cleaned/aggregated CSV out

In [2]:
import csv
from pathlib import Path

raw_csv = Path("data") / "sales_raw.csv"
rows = [
    {"product": "Widget", "region": "North", "units": "10", "price": "9.99"},
    {"product": "Widget", "region": "South", "units": "5",  "price": "9.99"},
    {"product": "Gadget", "region": "North", "units": "3",  "price": "19.99"},
    {"product": "Gadget", "region": "",      "units": "abc", "price": "19.99"},  # dirty row
    {"product": "Widget", "region": "North", "units": "7",  "price": "9.99"},
]
with raw_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["product", "region", "units", "price"])
    writer.writeheader()
    writer.writerows(rows)

print("Wrote raw CSV with", len(rows), "rows (including a dirty one).")


Wrote raw CSV with 5 rows (including a dirty one).


In [3]:
import csv
from pathlib import Path
from collections import defaultdict

raw_csv = Path("data") / "sales_raw.csv"
clean_csv = Path("data") / "sales_by_product.csv"

totals = defaultdict(float)
skipped = 0

with raw_csv.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            units = int(row["units"])
            price = float(row["price"])
            if not row["region"]:
                raise ValueError("missing region")
            totals[row["product"]] += units * price
        except (ValueError, KeyError):
            skipped += 1
            continue

with clean_csv.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["product", "total_revenue"])
    for product, revenue in sorted(totals.items()):
        writer.writerow([product, round(revenue, 2)])

print(f"Processed data, skipped {skipped} malformed row(s).")
print(clean_csv.read_text())


Processed data, skipped 1 malformed row(s).
product,total_revenue
Gadget,59.97
Widget,219.78



## Task 2 — Simple Web Scraper

In [4]:
# Install dependencies (uncomment if running for the first time)
# %pip install requests beautifulsoup4 --quiet
print("requires: requests, beautifulsoup4")


requires: requests, beautifulsoup4


In [5]:
import csv

URL = "https://quotes.toscrape.com/"   # a site built for scraping practice

def scrape_quotes(url):
    import requests
    from bs4 import BeautifulSoup
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    quotes = []
    for block in soup.select(".quote"):
        text = block.select_one(".text").get_text(strip=True)
        author = block.select_one(".author").get_text(strip=True)
        tags = [t.get_text(strip=True) for t in block.select(".tag")]
        quotes.append({"text": text, "author": author, "tags": ", ".join(tags)})
    return quotes

try:
    import requests
    from bs4 import BeautifulSoup
    quotes = scrape_quotes(URL)
    print(f"Scraped {len(quotes)} quotes from {URL}")
    for q in quotes[:3]:
        print("-", q["author"], "|", q["tags"])
except Exception as e:
    quotes = []
    print("Network access not available in this environment:", e)
    print("The scrape_quotes() function above will work in a normal environment with internet access.")


Network access not available in this environment: 403 Client Error: Forbidden for url: https://quotes.toscrape.com/
The scrape_quotes() function above will work in a normal environment with internet access.


In [6]:
from pathlib import Path
import csv

out_path = Path("data") / "scraped_quotes.csv"
if quotes:
    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["text", "author", "tags"])
        writer.writeheader()
        writer.writerows(quotes)
    print("Saved scraped data to", out_path)
else:
    print("No data scraped in this run (see note above) — rerun with internet access to populate the CSV.")


No data scraped in this run (see note above) — rerun with internet access to populate the CSV.


### Bonus: Paginated scraping (follow the "Next" link)

In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_all_pages(start_url, max_pages=5):
    all_quotes = []
    url = start_url
    for _ in range(max_pages):
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
        except Exception as e:
            print("Stopping - network error:", e)
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        for block in soup.select(".quote"):
            all_quotes.append(block.select_one(".text").get_text(strip=True))

        next_link = soup.select_one("li.next > a")
        if not next_link:
            break
        url = urljoin(url, next_link["href"])
    return all_quotes

# all_quotes = scrape_all_pages(URL)
# print(f"Collected {len(all_quotes)} quotes across pages")
print("Function defined - call scrape_all_pages(URL) in an environment with internet access.")


Function defined - call scrape_all_pages(URL) in an environment with internet access.
